# Optimal Transport for Domain Adaptation

In this notebook, we will discuss how **Optimal Transport (OT)** can be applied to **domain adaptation**, following the paper [[TPAMI 2016] *Optimal Transport for Domain Adaptation*](https://arxiv.org/pdf/1507.00504).

The notebook is organized as follows:

* First, we introduce the domain adaptation problem and explain why it is important.
* Then, we present the Optimal Transport formulation.
* Finally, we study the OT-based methods proposed in the paper to solve the domain adaptation problem.

---

# Domain Adaptation

Domain adaptation is a common problem in machine learning.

Suppose that we have a labeled dataset, called the **source domain**, that we use to train a machine learning model. Once deployed, the model will encounter new data, called the **target domain**, which may not follow the same distribution as the data used during training.

As a result, even if the model performs very well on the source domain, its performance may decrease significantly on the target domain.

For example, suppose we want to train a car detection model using videos recorded in one country. If we deploy this model in another country, the images may differ because of changes in lighting conditions, camera devices, viewpoints, weather conditions, or road environments. Although the model was trained correctly, these differences may cause it to perform poorly on the new data.

There are several domain adaptation settings:

* **Unsupervised domain adaptation:** no labels are available in the target domain.
* **Semi-supervised domain adaptation:** only a small number of target samples are labeled.
* **Supervised domain adaptation:** labels are available in both the source and target domains.

In this notebook, we focus on the most challenging and one of the most common settings: **unsupervised domain adaptation**.

---

# Optimal Transport

Let us suppose that the source and target domains are realizations of two random variables, $X_s$ and $X_t$, taking their values in the spaces $\omega_s$ and $\omega_t$, respectively.

We assume that the difference between the source and target domains is caused by a transformation

$$
T : \omega_s \rightarrow \omega_t
$$

such that

$$
P_s(y \mid x^s) = P_t(y \mid T(x^s)), \qquad \forall x^s \in \omega_s.
$$

This means that the probability for a source sample $x^s$ to belong to class $y$ is the same as the probability for its transformed version $T(x^s)$ to belong to the same class in the target domain.

Let $\mu_s$ and $\mu_t$ denote the probability distributions of $X_s$ and $X_t$, respectively.

Our goal is to find a transformation

$$
T : \omega_s \rightarrow \omega_t
$$

such that

$$
T_{\mu_s} = \mu_t.
$$

This means that, for every measurable set $A \subset \omega_t$,

$$
\mu_t(A) = \mu_s(T^{-1}(A)).
$$

You may wonder why this condition is important.

Let $A \subset \omega_t$ be a measurable set. Then,

$$
P(X_t \in A)= \mu_t(A)= \mu_s(T^{-1}(A))= P(X_s \in T^{-1}(A))= P(T(X_s) \in A).
$$

Therefore, the random variables $X_t$ and $T(X_s)$ follow the same probability distribution.

In other words, we are looking for a transformation that aligns the source and target distributions.

However, this is not enough.

Let

$$
c : \omega_s \times \omega_t \rightarrow \mathbb{R}^+
$$

be a cost function, where $c(x_i,x_j)$ represents the cost of transporting the source sample $x_i$ to the target sample $x_j$.

Among all the transformations that align the two distributions, we want the one with the smallest transportation cost.

The transportation cost of a transformation $T$ is defined by

$$
C(T)=\int_{\omega_s} c(x,T(x))d\mu_s(x).
$$

Minimizing this cost prevents the source domain from being excessively distorted during the transformation.

To summarize, we are looking for the following optimal transformation:

$$T^*= argmin\{C(T) =\int_{\omega_s} c(x,T(x))d\mu_s(x):T_{\mu_s}=\mu_t\}$$

In Optimal Transport theory, this transformation is called the **Monge map**, since it comes from the **Monge formulation** of the Optimal Transport problem.

The overall workflow is therefore:

1. Estimate the distributions $\mu_s$ and $\mu_t$ from the available data.
2. Compute the optimal transformation $T^*$.
3. Transform the source samples using $T^*$.
4. Train a classifier on the transformed source dataset $(T^*(X_s), Y_s)$.

Unfortunately, the optimal transformation $T^*$ does not always exist. For this reason, the **Kantorovich relaxation** was introduced.

The main idea is the following:

* In the **Monge formulation**, each source point is transported to exactly one target point.
* In the **Kantorovich formulation**, the mass of a source point can be split among several target points. This is known as **mass splitting**.

Let us now study the discrete Kantorovich formulation.


### Discrete Kantorovich formulation

Suppose that the source and target distributions are discrete:

$$
\mu_s = \sum_{i=1}^{n_s} p_i^s \delta_{x_i^s}
\qquad \text{and} \qquad
\mu_t = \sum_{j=1}^{n_t} p_j^t \delta_{x_j^t}.
$$

where

$$
p_i^s \ge 0,
\qquad
p_j^t \ge 0,
$$

and

$$
\sum_{i=1}^{n_s} p_i^s=\sum_{j=1}^{n_t} p_j^t =1.
$$

We denote

$$
p_s = (p_1^s,\ldots,p_{n_s}^s)
\qquad \text{and} \qquad
p_t = (p_1^t,\ldots,p_{n_t}^t).
$$

Here, $\delta_{x_i}$ denotes the Dirac measure, defined by

$$
\delta_{x_i}(A)=
\begin{cases}
1 & \text{if } x_i \in A, \
0 & \text{otherwise.}
\end{cases}
$$

Let

$$
C \in \mathbb{R}_+^{n_s \times n_t}
$$

be the transportation cost matrix, where $C_{i,j}$ represents the cost of transporting mass from the source point $x_i^s$ to the target point $x_j^t$.

In most applications, we simply choose

$$
C_{i,j} = ||x_i^s - x_j^t||_2^2.
$$

Instead of looking for a transformation, the Kantorovich formulation looks for a **transport plan**, represented by a matrix

$$
\gamma \in U(p_s,p_t),
$$

where

$$
U(p_s,p_t) = \{\gamma \in \mathbb{R}_+^{n_s \times n_t}:\gamma \mathbf{1}_{n_t}=p_s,\gamma^T \mathbf{1}_{n_s}=p_t\}.
$$

Each coefficient $\gamma_{i,j}$ represents the amount of mass transported from the source point $x_i^s$ to the target point $x_j^t$.

The first constraint,

$$
\gamma \mathbf{1}_{n_t}=p_s,
$$

is equivalent to

$$
\sum_j \gamma_{i,j}=p_i^s,
\qquad
\forall i.
$$

This means that the total mass leaving the source point $x_i^s$ must be equal to its initial mass $p_i^s$.

Similarly, the second constraint,

$$
\gamma^T \mathbf{1}_{n_s}=p_t,
$$

is equivalent to

$$
\sum_i \gamma_{i,j}=p_j^t,
\qquad
\forall j.
$$

This means that the total mass arriving at the target point $x_j^t$ must be equal to its mass $p_j^t$.

These two constraints simply express the **conservation of mass**.

The transportation cost associated with a transport plan is

$$
\min_{\gamma \in U(p_s,p_t)} \langle \gamma, C \rangle= \min_{\gamma \in U(p_s,p_t)}\sum_{i,j}\gamma_{i,j} C_{i,j}.
$$

The optimal transport plan is therefore

$$
\gamma^*=\operatorname*{argmin}_{\gamma \in U(p_s,p_t)}\langle \gamma, C \rangle.
$$

Once the optimal transport plan has been computed, we can transform the source domain.

For a source point $x_i^s$, the transformed point $T(x_i^s)$ is obtained by minimizing the transportation cost induced by its transported mass:

$$
T(x_i^s)=\operatorname*{argmin}\sum_j\gamma_{i,j}C_{i,j}.
$$

When the cost function is the squared Euclidean distance,

$$
C_{i,j}=||x_i^s-x_j^t||_2^2,
$$

it can be shown that the solution is simply the barycenter of the target points weighted by the transported mass:

$$
T(x_i^s)=\frac{\sum_j \gamma_{i,j}x_j^t}{\sum_j \gamma_{i,j}}.
$$

Using matrix notation, the transformation of the whole source domain can be written as

$$
T(X_s)=diag(\gamma^*\mathbf{1}_{n_t})^{-1}\gamma^*X_t.
$$

A proof of this result is available here:

[Proof 1](https://github.com/Samuel-Vangu/domain-adaptation-with-optimal-transport/blob/main/src/tpami2016_optimal_transport_domain_adaptation/notebooks/proofs/proof_1.pdf)

Moreover, when both distributions are uniform,

$$
p_i^s=\frac{1}{n_s}
\qquad \text{and} \qquad
p_j^t=\frac{1}{n_t},
$$

the previous expression simplifies to

$$
T(X_s)=n_s \cdot \gamma^*X_t.
$$

A proof of this result is available here:

[Proof 2](https://github.com/Samuel-Vangu/domain-adaptation-with-optimal-transport/blob/main/src/tpami2016_optimal_transport_domain_adaptation/notebooks/proofs/proof_2.pdf)

# Regularization Techniques

The transport plan obtained with the standard Optimal Transport formulation only minimizes the transportation cost. However, in domain adaptation, we often have additional information that can help compute a better transport plan.

The paper introduces three regularization techniques:

* Entropic regularization
* Group Lasso regularization
* Laplacian regularization

---

## Entropic Regularization

The first regularization consists of adding an entropy term to the original Optimal Transport problem.

The optimization problem becomes

$$
\gamma^*= \operatorname*{argmin}_{\gamma \in U(p_s,p_t)}\langle \gamma, C \rangle-\lambda h(\gamma),
$$

where

$$
h(\gamma)=-\sum_{i,j}\gamma_{i,j}\log(\gamma_{i,j})
$$

is the entropy of the transport plan, and $\lambda \ge 0$ is the regularization parameter.

To minimize the objective function, we must simultaneously:

* minimize the transportation cost $\langle \gamma, C \rangle$,
* maximize the entropy $h(\gamma)$.

Intuitively, a transport plan with a larger entropy distributes its mass more evenly over the matrix. Instead of concentrating almost all the transported mass on only a few coefficients, the mass is spread over more entries.

This regularization has several advantages. In particular, it makes the optimization problem more numerically stable and allows it to be solved efficiently using the **Sinkhorn algorithm**.

---

## Group Lasso Regularization

The source labels can also be used to improve the transport plan.

The main idea is to discourage transport plans in which source samples from different classes send their mass to the same target samples.

Ideally, each target sample should receive most of its mass from source samples belonging to the same class.

The optimization problem becomes

$$
\gamma^*= \operatorname*{argmin}_{\gamma \in U(p_s,p_t)}\langle \gamma, C \rangle-\lambda h(\gamma)+\eta \rho(\gamma),
$$

where $\eta \ge 0$ is another regularization parameter and $\rho(\gamma)$ is defined as

$$
\rho(\gamma)=\sum_j\sum_{cl}||\gamma(I_{cl},j)||_2.
$$

Here,

* $I_{cl}$ is the set of indices of the source samples belonging to class $cl$;
* $\gamma(I_{cl},j)$ is the vector containing all the transport coefficients in column $j$ corresponding to class $cl$.

Its Euclidean norm is

$$||\gamma(I_{cl},j)||_2=\sqrt{\sum_{i \in I_{cl}}(\gamma_{i,j})^2}.$$

Suppose that a target sample receives mass from several different classes.

In that case, many terms of the form

$$||\gamma(I_{cl},j)||_2
$$

will contribute to the sum, making the regularization term larger.

Since we minimize the objective function, this regularization penalizes transport plans where target samples receive mass from multiple classes.

This regularization assumes that the class distributions are identical, or at least very similar, in the source and target domains.

---

## Laplacian Regularization

The goal of Laplacian regularization is to preserve the geometric structure of the source domain during the transportation process.

In other words, if two source samples are similar before transportation, then their transformed versions should also remain similar.

To measure the similarity between source samples, we define a similarity matrix

$$
S \in \mathbb{R}^{n_s \times n_s},
$$

where $S_{i,j}$ measures the similarity between the source samples $x_i^s$ and $x_j^s$.

A common choice is

$$
S_{i,j}=\exp(\frac{||x_i^s-x_j^s||_2^2}{\sigma^2}).$$

To preserve the class structure, we additionally set

$$
S_{i,j}=0
$$

whenever $x_i^s$ and $x_j^s$ do not belong to the same class.

The optimization problem becomes

$$
\gamma^*= \operatorname*{argmin}_{\gamma \in U(p_s,p_t)}\langle \gamma, C \rangle-\lambda h(\gamma)+\eta \rho(\gamma),
$$

where

$$
\rho(\gamma)=\frac{1}{n_s^2}
\sum_{i,j}S_{i,j}||T(x_i^s)-T(x_j^s)||_2^2.
$$

### Interpretation

If $S_{i,j}$ is large, then the source samples $x_i^s$ and $x_j^s$ are very similar. Therefore, we want their transformed versions to remain close, meaning that

$$
||
T(x_i^s)-T(x_j^s)
||_2^2
$$

should be small.

On the other hand, if $S_{i,j}$ is close to zero, then the similarity between these two source samples is weak. In this case, the distance between their transformed versions has little influence on the objective function.

As a result, this regularization helps preserve the local geometry of the source domain while aligning it with the target domain.



# Conclusion

In this notebook, we introduced the domain adaptation problem and showed how **Optimal Transport** provides a principled framework for aligning the source and target distributions.

We first presented the **Monge formulation**, whose goal is to find an optimal transformation that aligns both distributions while minimizing the transportation cost. Since such a transformation does not always exist, we introduced the **Kantorovich formulation**, which relaxes the problem by allowing the mass of each source sample to be transported to multiple target samples.

We then studied the discrete Optimal Transport problem and showed how the optimal transport plan can be used to transform the source domain. Finally, we presented three regularization techniques proposed in the paper. These regularizations incorporate additional information, such as the source labels or the geometric structure of the data, in order to compute transport plans that are better suited for domain adaptation.

Overall, Optimal Transport offers a natural and mathematically grounded approach to reducing the distribution shift between the source and target domains while preserving important properties of the data. This makes it a powerful tool for addressing unsupervised domain adaptation problems.
